In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType

In [2]:
# Spark master
# spark = (
#     SparkSession.builder
#     .appName("yahoo_finance")
#     .master("spark://localhost:7077")
#     .getOrCreate()
# )

# Spark connect
spark = (
    SparkSession.builder
    .appName("yahoo_finance")
    .remote("sc://localhost:15002")
    .getOrCreate()
)

/Users/fahmialfareza/Code/learning/data-engineering/fullstack-bangalore-advanced-data-engineering/17-adv-data-engineering/.venv/lib/python3.11/site-packages/pyspark/sql/connect/utils.py:41: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/Users/fahmialfareza/Code/learning/data-engineering/fullstack-bangalore-advanced-data-engineering/17-adv-data-engineering/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
schema = StructType() \
    .add("symbol", StringType()) \
    .add("date", StringType()) \
    .add("price", DoubleType()) \
    .add("return", DoubleType()) \
    .add("volatility", DoubleType())

In [4]:
df_kafka = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "yahoo_finance")
    .option("startingOffsets", "latest")
    .load()
)

In [5]:
df_final = (
    df_kafka
    .select(
        from_json(
            col("value").cast("string"),
            schema
        ).alias("data")
    )
    .filter(col("data").isNotNull())
    .select("data.*")
)

In [6]:
# Only for Spark Master
# spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://localhost:9001")
# spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", "minio")
# spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "minio123")
# spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
# spark._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

# Only for Spark Connect
df = spark.range(10)

df.write.mode("overwrite").parquet(
    "s3a://data-lake/test-spark-connect"
)

In [7]:
df_final.writeStream \
    .format("parquet") \
    .option("path", "s3a://data-lake/curated/yahoo_finance") \
    .option("checkpointLocation", "/tmp/stock_cp") \
    .outputMode("append") \
    .start()